In [7]:
#A
from pymongo import MongoClient

client = MongoClient("mongodb://localhost:27017/")
db = client["kopi_nusantara"]
menu = db["menu"]

menu.delete_many({})

menu.insert_many([
    {"nama_menu": "Espresso", "harga": 18000, "kategori": "Kopi", "tingkat_kepahitan": 5, "stok": 50},
    {"nama_menu": "Americano", "harga": 20000, "kategori": "Kopi", "tingkat_kepahitan": 4, "stok": 60},
    {"nama_menu": "Cappuccino", "harga": 25000, "kategori": "Kopi", "tingkat_kepahitan": 3, "stok": 70},
    {"nama_menu": "Latte", "harga": 27000, "kategori": "Kopi", "tingkat_kepahitan": 2, "stok": 65},
    {"nama_menu": "V60 Arabika Gayo", "harga": 32000, "kategori": "Kopi", "tingkat_kepahitan": 4, "stok": 30, "rekomendasi_barista": True},
    {"nama_menu": "Kopi Susu Gula Aren", "harga": 22000, "kategori": "Kopi", "tingkat_kepahitan": 2, "stok": 100},
    {"nama_menu": "Vietnam Drip", "harga": 24000, "kategori": "Kopi", "tingkat_kepahitan": 5, "stok": 40},
    {"nama_menu": "Nasi Goreng Kampung", "harga": 28000, "kategori": "Makanan", "pedas_level": 2, "stok": 40},
    {"nama_menu": "Mie Goreng Pedas", "harga": 26000, "kategori": "Makanan", "pedas_level": 3, "stok": 45},
    {"nama_menu": "Roti Bakar Cokelat", "harga": 15000, "kategori": "Makanan", "pedas_level": 0, "stok": 55},
    {"nama_menu": "Kentang Goreng", "harga": 20000, "kategori": "Makanan", "pedas_level": 1, "stok": 60},
    {"nama_menu": "Ayam Geprek", "harga": 30000, "kategori": "Makanan", "pedas_level": 3, "stok": 35, "rekomendasi_barista": False},
    {"nama_menu": "Es Teh Manis", "harga": 8000, "kategori": "Minuman", "stok": 120},
    {"nama_menu": "Matcha Latte", "harga": 27000, "kategori": "Minuman", "stok": 50},
])

print("Jumlah dokumen:", menu.count_documents({}))

Jumlah dokumen: 14


In [8]:
#B
print("1. Kopi dengan tingkat_kepahitan >3:")
for m in menu.find({"kategori":"Kopi", "tingkat_kepahitan": {"$gt": 3}}, {"_id": 0}):
    print("  ", m)

print("\n2. Menu dengan harga antara 15000 dan 30000:")
for m in menu.find({"harga": {"$gt":15000, "$lte":30000}}, {"_id":0}):
    print("  ", m)

print("\n3. MEnu yang tidak punya field pedas_level")
for m in menu.find({"pedas_level": {"$exists": False}}, {"_id": 0}):
    print("  ", m)

1. Kopi dengan tingkat_kepahitan >3:
   {'nama_menu': 'Espresso', 'harga': 18000, 'kategori': 'Kopi', 'tingkat_kepahitan': 5, 'stok': 50}
   {'nama_menu': 'Americano', 'harga': 20000, 'kategori': 'Kopi', 'tingkat_kepahitan': 4, 'stok': 60}
   {'nama_menu': 'V60 Arabika Gayo', 'harga': 32000, 'kategori': 'Kopi', 'tingkat_kepahitan': 4, 'stok': 30, 'rekomendasi_barista': True}
   {'nama_menu': 'Vietnam Drip', 'harga': 24000, 'kategori': 'Kopi', 'tingkat_kepahitan': 5, 'stok': 40}

2. Menu dengan harga antara 15000 dan 30000:
   {'nama_menu': 'Espresso', 'harga': 18000, 'kategori': 'Kopi', 'tingkat_kepahitan': 5, 'stok': 50}
   {'nama_menu': 'Americano', 'harga': 20000, 'kategori': 'Kopi', 'tingkat_kepahitan': 4, 'stok': 60}
   {'nama_menu': 'Cappuccino', 'harga': 25000, 'kategori': 'Kopi', 'tingkat_kepahitan': 3, 'stok': 70}
   {'nama_menu': 'Latte', 'harga': 27000, 'kategori': 'Kopi', 'tingkat_kepahitan': 2, 'stok': 65}
   {'nama_menu': 'Kopi Susu Gula Aren', 'harga': 22000, 'kategori':

In [9]:
#C
menu.update_many({}, {"$set": {"status":"Tersedia"}})

menu.update_one({"nama_menu":"Espresso"},{"$set":{"status":"Habis"}})
menu.update_one({"nama_menu":"Vietnam_Drip"},{"$set":{"status":"Habis"}})

for m in menu.find({}, {"_id": 0, "nama_menu": 1, "status":1}):
    print(f" - {m['nama_menu']}: {m['status']}")

 - Espresso: Habis
 - Americano: Tersedia
 - Cappuccino: Tersedia
 - Latte: Tersedia
 - V60 Arabika Gayo: Tersedia
 - Kopi Susu Gula Aren: Tersedia
 - Vietnam Drip: Tersedia
 - Nasi Goreng Kampung: Tersedia
 - Mie Goreng Pedas: Tersedia
 - Roti Bakar Cokelat: Tersedia
 - Kentang Goreng: Tersedia
 - Ayam Geprek: Tersedia
 - Es Teh Manis: Tersedia
 - Matcha Latte: Tersedia


In [10]:
#D
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, StringType, LongType
from pyspark.sql.functions import avg, count

spark = SparkSession.builder \
    .appName("Tugas9-KopiNusantara") \
    .master("local[*]") \
    .getOrCreate()
spark.sparkContext.setLogLevel("ERROR")
dokumen = list(menu.find({}))
schema = StructType([
    StructField("_id", StringType(), True),
    StructField("nama_menu", StringType(), True),
    StructField("harga", LongType(), True),
    StructField("kategori", StringType(), True),
    StructField("tingkat_kepahitan", LongType(), True),
    StructField("pedas_level", LongType(), True),
    StructField("stok", LongType(), True),
    StructField("status", StringType(), True),
])
rows = [
    (
        str(d["_id"]),
        d.get("nama_menu"),
        d.get("harga"),
        d.get("kategori"),
        d.get("tingkat_kepahitan"),
        d.get("pedas_level"),
        d.get("stok"),
        d.get("status"),
    )
    for d in dokumen
]

df_menu = spark.createDataFrame(rows, schema=schema)
print("1. printSchema():")
df_menu.printSchema()
df_menu.show(truncate=False)
print("2. Rata-rata harga per kategori:")
df_menu.groupBy("kategori").agg(
    avg("harga").alias("rata_rata_harga"),
    count("nama_menu").alias("jumlah_menu")
).show()

spark.stop()

1. printSchema():
root
 |-- _id: string (nullable = true)
 |-- nama_menu: string (nullable = true)
 |-- harga: long (nullable = true)
 |-- kategori: string (nullable = true)
 |-- tingkat_kepahitan: long (nullable = true)
 |-- pedas_level: long (nullable = true)
 |-- stok: long (nullable = true)
 |-- status: string (nullable = true)



+------------------------+-------------------+-----+--------+-----------------+-----------+----+--------+
|_id                     |nama_menu          |harga|kategori|tingkat_kepahitan|pedas_level|stok|status  |
+------------------------+-------------------+-----+--------+-----------------+-----------+----+--------+
|6ae5f1db19a0d5acc297fc62|Espresso           |18000|Kopi    |5                |NULL       |50  |Habis   |
|6ae5f1db19a0d5acc297fc63|Americano          |20000|Kopi    |4                |NULL       |60  |Tersedia|
|6ae5f1db19a0d5acc297fc64|Cappuccino         |25000|Kopi    |3                |NULL       |70  |Tersedia|
|6ae5f1db19a0d5acc297fc65|Latte              |27000|Kopi    |2                |NULL       |65  |Tersedia|
|6ae5f1db19a0d5acc297fc66|V60 Arabika Gayo   |32000|Kopi    |4                |NULL       |30  |Tersedia|
|6ae5f1db19a0d5acc297fc67|Kopi Susu Gula Aren|22000|Kopi    |2                |NULL       |100 |Tersedia|
|6ae5f1db19a0d5acc297fc68|Vietnam Drip       |

KESIMPULAN

Tantangan utama saat mengonversi data MongoDB ke SPark DataFrame adalah ketidakseragaan field antar dokumen. Di MOngoDB, menu kopi punya tingkah kepahitan sedangkan menu makanan punya pedas level, dan dokumen lain tidak punya keduanya. Spark DataFrame membutuhkan skema tabel yang seragam. Sehingga setiap kolom harus punya tipe data yang jelas. Kemudian untuk tantangan lainnya adalah tipe khusus MongoDB seperi id yang bertipe objek (yang tidak dikenali Spark), sehingga harus diubah menjadi string. JUga ketika memakai pandas, kolom integer yang punya nilai kosong berubah menjadi float denga NaN, bukan NULL. Lalu masalah ini diatasi dengan membuat skema StructType secara eksplisit. kemudian membangun baris data dengan d.get("field") agar field ang tidak ada otomatis menjadi none atau null. DEngan cara ini semua field dari seluruh dokumen menjadi kolom, dan dkokumen yang tidak memilikinya berisi null. Hasilnya analisis groupby tetap berjalan lancar. Sekian.
